# 📓 01 — Getting Started with Strands RobotsLearn the package structure, installation options, and how the lazy-loading import system works.

In [ ]:
# Install strands-robots (uncomment the extras you need)# !pip install strands-robots                    # base (policies + tools)# !pip install "strands-robots[sim-mujoco]"      # + MuJoCo simulation# !pip install "strands-robots[groot-service]"   # + GR00T ZMQ client# !pip install "strands-robots[lerobot]"         # + LeRobot direct inference# !pip install "strands-robots[all]"             # everything

## Package Structure```strands_robots/├── __init__.py        # Thin — lazy loads heavy modules on first access├── robot.py           # Robot AgentTool (async control, status, stop)├── utils.py           # require_optional(), path helpers├── policies/          # Policy abstraction layer├── simulation/        # MuJoCo (+ future Isaac, Newton)├── registry/          # JSON-driven robot & policy discovery├── tools/             # Strands @tool functions└── assets/            # Menagerie model resolution```

## Lazy Loading`strands_robots` uses `__getattr__` on the package to defer heavy imports (torch, lerobot, mujoco)until the moment you actually access them. This means `import strands_robots` is always fast.

In [ ]:
import strands_robotsimport time# These are always available immediately (no heavy deps)print("Light imports (instant):")print(f"  Policy ABC:    {strands_robots.Policy}")print(f"  MockPolicy:    {strands_robots.MockPolicy}")print(f"  create_policy: {strands_robots.create_policy}")

In [ ]:
# Heavy symbols are lazy — only loaded on first access# This would trigger: import lerobot, torch, numpy, etc.# Uncomment to test:# robot_cls = strands_robots.Robot  # triggers lerobot import# groot_cls = strands_robots.Gr00tPolicy  # triggers numpy import# You can check what's available without importing:print("\nAll public symbols:")print(strands_robots.__all__)

## require_optional() — Consistent Dependency ManagementThe `require_optional()` utility provides clear error messages when optional deps are missing.

In [ ]:
from strands_robots.utils import require_optional# This will work if numpy is installed (it's a base dep)np = require_optional("numpy")print(f"numpy loaded: {np.__version__}")# This would give a helpful error if zmq is missing:try:    zmq = require_optional(        "zmq",        pip_install="pyzmq",        extra="groot-service",        purpose="GR00T service inference"    )    print(f"zmq loaded: {zmq.__version__}")except ImportError as e:    print(f"Expected error:\n{e}")

## Path ResolutionAll asset paths go through `utils.py` — one source of truth.

In [ ]:
from strands_robots.utils import get_assets_dir, get_base_dir, resolve_asset_pathprint(f"Base dir:   {get_base_dir()}")print(f"Assets dir: {get_assets_dir()}")# resolve_asset_path handles relative, absolute, and None pathsprint(f"\nRelative:  {resolve_asset_path('my_robot')}")print(f"Absolute:  {resolve_asset_path('/tmp/model.xml')}")print(f"Default:   {resolve_asset_path(None, 'so100')}")print(f"\nOverride with: export STRANDS_ASSETS_DIR=/custom/path")